In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc311_cleaned.csv", parse_dates=['created_date', 'closed_date'])
print(df.shape)
df.head()

In [ ]:
print(df.dtypes[['created_date', 'closed_date']])

In [ ]:
df['response_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600

print(df['response_hours'].describe())

In [ ]:
invalid_response = (df['response_hours'] < 0).sum()
print(f"Rows with negative response time: {invalid_response}")

# Negative response time is a data error (closed before created) — remove these
df = df[(df['response_hours'] >= 0) | (df['response_hours'].isna())]
print(df.shape)

In [ ]:
# Cap at 90 days, consistent with EDA
df['response_hours_capped'] = df['response_hours'].clip(upper=90*24)

In [ ]:
df['response_hours_log'] = np.log1p(df['response_hours_capped'])

print(df[['response_hours', 'response_hours_capped', 'response_hours_log']].describe())

In [ ]:
import plotly.express as px
fig = px.histogram(df, x='response_hours_log', nbins=50, title='Log-Transformed Response Time')
fig.show()

In [ ]:
df['is_resolved'] = df['closed_date'].notna().astype(int)

print(df['is_resolved'].value_counts())

In [ ]:
df['day_of_week'] = df['created_date'].dt.day_name()
df['day_of_week_num'] = df['created_date'].dt.dayofweek  # 0=Monday, 6=Sunday

print(df['day_of_week'].value_counts())

In [ ]:
df['is_weekend'] = df['day_of_week_num'].isin([5, 6]).astype(int)

print(df['is_weekend'].value_counts())

In [ ]:
df['hour_of_day'] = df['created_date'].dt.hour

print(df['hour_of_day'].value_counts().sort_index())

In [ ]:
fig = px.histogram(df, x='hour_of_day', nbins=24, title='Complaints by Hour of Day')
fig.show()

In [ ]:
df['month'] = df['created_date'].dt.month

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)

print(df['season'].value_counts())

In [ ]:
high_priority_keywords = ['emergency', 'gas', 'fire', 'water main', 'structural', 'collapse']
medium_priority_keywords = ['noise', 'illegal', 'blocked', 'leak']

def assign_priority(complaint_type):
    text = str(complaint_type).lower()
    if any(k in text for k in high_priority_keywords):
        return 'High'
    elif any(k in text for k in medium_priority_keywords):
        return 'Medium'
    else:
        return 'Low'

df['priority'] = df['complaint_type'].apply(assign_priority)

print(df['priority'].value_counts())

In [ ]:
print(df['module'].value_counts())

Borough was found to be a sufficient and clean geographic feature based on EDA — the 5 boroughs form distinct, non-overlapping spatial clusters, so additional unsupervised geo-clustering (e.g., k-means) was not necessary

In [ ]:
# One-hot encode low-cardinality categoricals
categorical_cols = ['borough', 'module', 'priority', 'season', 'submission_channel']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)
print(df_encoded.columns.tolist())

In [ ]:
# One-hot encode low-cardinality categoricals
categorical_cols = ['borough', 'module', 'priority', 'season', 'submission_channel']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)
print(df_encoded.columns.tolist())

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/nyc311_features.csv")
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())

In [ ]:
final_features = [
    'unique_key', 'created_date', 'closed_date', 'complaint_type',
    'response_hours', 'response_hours_capped', 'response_hours_log',
    'is_resolved', 'day_of_week', 'day_of_week_num', 'is_weekend',
    'hour_of_day', 'month', 'season', 'priority', 'module', 'borough',
    'latitude', 'longitude', 'zip_code', 'status', 'agency'
]

df_final = df[final_features].copy()

df_final.to_csv("../data/processed/nyc311_features.csv", index=False)
print("Saved:", df_final.shape)